In [ ]:
import json

import openeo.processes
from utils import urls, utils

import openeo

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
with open("../AOI/upstream_and_downstream.geojson", "r") as f:
    upstream_and_downstream = json.load(f)

In [ ]:
start_date = "2018-01-01"
end_date = "2025-12-31"

In [ ]:
baseline_start = "2018-01-01"
baseline_end = "2022-12-31"

In [ ]:
water_occurrence_threshold = 10  # units: %

In [ ]:
# for a pixel to be considered anomalous,
# its anomaly must be >= anomalyZThreshold
# and its value must be >= effectThreshold * climatological median

anomalyZThreshold = 2
effectThreshold = 0.40

In [ ]:
# number of months in a year that must be anomalous to be a hotspot
minMonthsPerYearForHotspot = 2

# min connected area of hotspot pixels (8-connectivity)
minContiguousPixels = 6

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

Sentinel 2 L2A

In [ ]:
optical_bands = ["B03", "B04", "B05"]

In [ ]:
S2 = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=upstream_and_downstream,
    temporal_extent=[
        start_date,
        end_date,
    ],
    bands=optical_bands + ["SCL"],
)

In [ ]:
# where we have a long time dimension,
# extract a single time slice for any intermediate results

DEBUG_TIME_INDEX = 1

In [ ]:
_slice = utils.t_index(S2, index=DEBUG_TIME_INDEX)

process_graph_results.append(
    _slice.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0010_S2",
        },
    )
)
process_graph_results.append(
    _slice.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_S2",
        },
    )
)

mask Sentinel 2 based on SCL

In [ ]:
scl = S2.band("SCL")
scl_mask = (
    (scl == 3)  # cloud shadow
    | (scl == 8)  # medium probability cloud
    | (scl == 9)  # high probability cloud
    | (scl == 10)  # cirrus
    | (scl == 11)  # snow or ice
)

In [ ]:
_slice = utils.t_index(scl_mask, index=DEBUG_TIME_INDEX)

process_graph_results.append(
    _slice.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_scl_mask",
        },
    )
)

In [ ]:
S2 = S2.filter_bands(optical_bands).mask(scl_mask)

In [ ]:
_slice = utils.t_index(S2, index=DEBUG_TIME_INDEX)

process_graph_results.append(
    _slice.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0030_S2",
        },
    )
)
process_graph_results.append(
    _slice.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_S2",
        },
    )
)

persistent water mask

In [ ]:
gsw = connection.load_stac(
    urls.JRC_GSW_STAC,
    spatial_extent=upstream_and_downstream,
)

In [ ]:
process_graph_results.append(
    gsw.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0040_gsw",
        },
    )
)

In [ ]:
# drop time dimension (length 1)
gsw = gsw.drop_dimension("t")

In [ ]:
water_extent_mask = gsw.band("occurrence") >= water_occurrence_threshold

In [ ]:
process_graph_results.append(
    water_extent_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0050_water_extent_mask",
        },
    )
)

In [ ]:
# reproject and resample onto S2
water_extent_mask = water_extent_mask.resample_cube_spatial(S2)

In [ ]:
process_graph_results.append(
    water_extent_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0060_water_extent_mask",
        },
    )
)

In [ ]:
# invert after resampling to avoid edge effects
inverse_water_extent_mask = utils.logical_not(water_extent_mask)

In [ ]:
process_graph_results.append(
    inverse_water_extent_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0070_inverse_water_extent_mask",
        },
    )
)

mask Sentinel 2 based on persistent water mask

In [ ]:
S2 = S2.mask(inverse_water_extent_mask)

In [ ]:
_slice = utils.t_index(S2, index=DEBUG_TIME_INDEX)

process_graph_results.append(
    _slice.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0080_S2",
        },
    )
)
process_graph_results.append(
    _slice.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0080_S2",
        },
    )
)

computes indicies

- green = B03
- red = B04
- red_edge = B05

NDTI = (red - green) / (red + green)

NDCI = (red_edge - red) / (red_edge + red)

In [ ]:
green = S2.band("B03")
red = S2.band("B04")
red_edge = S2.band("B05")
ndti = red.normalized_difference(green).add_dimension("bands", "NDTI", type="bands")
ndci = red_edge.normalized_difference(red).add_dimension("bands", "NDCI", type="bands")
indices = ndti.merge_cubes(ndci)

In [ ]:
_slice = utils.t_index(indices, index=DEBUG_TIME_INDEX)

process_graph_results.append(
    _slice.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0090_indices",
        },
    )
)
process_graph_results.append(
    _slice.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0090_indices",
        },
    )
)

monthly median composite

In [ ]:
monthly_median_composite = indices.aggregate_temporal_period(
    period="month",
    reducer=openeo.processes.median,
)

In [ ]:
process_graph_results.append(
    monthly_median_composite.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0100_monthly_median_composite",
        },
    )
)

Climatology

median and standard deviation

In [ ]:
climatology_udf_context = {
    # NDTI is used as the primary variable for baseline construction,
    # anomaly detection and annual hotspot generation
    "target_band": "NDTI",
    "baseline_start": baseline_start,
    "baseline_end": baseline_end,
    "std_min": 0.0001,
}

climatology_udf = openeo.UDF.from_file(
    "./udf/monthly_climatology.py",
    runtime="Python",
    version="3.11",
    context=climatology_udf_context,
)

In [ ]:
climatology = monthly_median_composite.apply_dimension(
    process=climatology_udf,
    dimension="t",
)

In [ ]:
# ⚠️ openEO python client does not run UDF apply_metadata()!
# so we have to do it manually here

import udf.monthly_climatology

climatology.metadata = udf.monthly_climatology.apply_metadata(
    climatology.metadata, climatology_udf_context
)

In [ ]:
process_graph_results.append(
    climatology.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0110_climatology",
        },
    )
)

calculate anomalies

In [ ]:
Z = (climatology.band("NDTI") - climatology.band("NDTI_median")) / climatology.band(
    "NDTI_std"
)

n_valid_months

For each year, count the number of months in which we have a valid value for Z.

ie. that month was not (cloud) masked.

In [ ]:
n_valid_months = Z.aggregate_temporal_period(
    period="year",
    reducer=openeo.processes.count,
)

In [ ]:
_tmp = n_valid_months.add_dimension("bands", "n_valid_months", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0115_n_valid_months",
        },
    )
)

exceedance event in month

- Z >= `anomalyZThreshold`
- X >= (1 + `effectThreshold`) * median

In [ ]:
# merge the climatology back in, so that we can do band math
Z = Z.add_dimension("bands", "NDTI_Z", type="bands")
Z = Z.merge_cubes(climatology)

In [ ]:
process_graph_results.append(
    Z.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0120_Z",
        },
    )
)

In [ ]:
# does not propagate nodata
# pixels previously masked by `inverse_water_extent_mask` are set to 0
exceedance = (Z.band("NDTI_Z") >= anomalyZThreshold) & (
    Z.band("NDTI") >= (1 + effectThreshold) * Z.band("NDTI_median")
)

In [ ]:
_tmp = exceedance.add_dimension("bands", "NDTI_exceedance", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0130_exceedance",
        },
    )
)

exceedance_count = yearly sum of montly exceedance events

In [ ]:
# ⚠️ https://forum.dataspace.copernicus.eu/t/aggregate-temporal-period-with-sum-behaves-like-xor/5667
# exceedance has dtype bool: we can't sum that

exceedance_count = (1.0 * exceedance).aggregate_temporal_period(
    period="year",
    reducer=openeo.processes.sum,
)

In [ ]:
_tmp = exceedance_count.add_dimension("bands", "NDTI_exceedance_count", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0140_exceedance_count",
        },
    )
)

A pixel is a hotspot (in a given year) if exceedance_count >= minMonthsPerYearForHotspot

In [ ]:
hotspot = exceedance_count >= minMonthsPerYearForHotspot

In [ ]:
_tmp = hotspot.add_dimension("bands", "NDTI_hotspot", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0150_hotspot",
        },
    )
)

apply connected component constraint minContiguousPixels

In [ ]:
connectivity_filter_udf_context = {
    "min_pixels": minContiguousPixels,
}

connectivity_filter_udf = openeo.UDF.from_file(
    "./udf/connectivity_filter.py",
    runtime="Python",
    version="3.11",
    context=connectivity_filter_udf_context,
)

In [ ]:
# datacube returned from UDF has float dtype

hotspot = hotspot.apply_neighborhood(
    process=connectivity_filter_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 8, "unit": "px"},
        {"dimension": "y", "value": 8, "unit": "px"},
    ],
)

In [ ]:
_tmp = hotspot.add_dimension("bands", "NDTI_hotspot", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0160_hotspot",
        },
    )
)

hotspot_exceedance_count = exceedance_count only for hotspot pixels

In [ ]:
inverse_hotspot = utils.logical_not(hotspot)

In [ ]:
_tmp = inverse_hotspot.add_dimension("bands", "inverse_hotspot", type="bands")

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0180_inverse_hotspot",
        },
    )
)

In [ ]:
hotspot_exceedance_count = exceedance_count.mask(inverse_hotspot)

In [ ]:
_tmp = hotspot_exceedance_count.add_dimension(
    "bands", "hotspot_exceedance_count", type="bands"
)

process_graph_results.append(
    _tmp.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0190_hotspot_exceedance_count",
        },
    )
)

Generate the water_valid layer (as in the GEE reference implementation).

Defined as any pixel where we have a least 1 month of data in that year.

In [ ]:
water_valid = n_valid_months > 0

create the final raster, with bands:

- hotspot (pixel mask)
- hotspot_exceedance_count (0-12, masked)
- n_valid_months (0–12, masked)
- water_valid (pixel mask)

This raster has dtype float.

In [ ]:
b1 = hotspot.add_dimension("bands", "hotspot", type="bands")
b2 = hotspot_exceedance_count.add_dimension(
    "bands", "hotspot_exceedance_count", type="bands"
)
b3 = n_valid_months.add_dimension("bands", "n_valid_months", type="bands")
b4 = water_valid.add_dimension("bands", "water_valid", type="bands")
final_raster = b1.merge_cubes(b2).merge_cubes(b3).merge_cubes(b4)

In [ ]:
process_graph_results.append(
    final_raster.save_result(
        format="netCDF",
        options={
            "filename_prefix": "0200_final_raster",
        },
    )
)

aggregate over upstream and downstream geometries

count the number of hotspot pixels and the number of water_valid pixels per year.

In [ ]:
def reducer(data: openeo.processes.ProcessBuilder) -> openeo.processes.ProcessBuilder:
    """sum, count, mean"""
    return openeo.processes.array_create(
        [
            data.sum(),
            data.count(),
            data.mean(),
        ]
    )


vector_stats = final_raster.filter_bands(
    ["hotspot", "hotspot_exceedance_count", "water_valid"]
).aggregate_spatial(
    geometries=upstream_and_downstream,
    reducer=reducer,
)

In [ ]:
# download Parquet, because it is the only vector format
# that returns both geometries and properties

process_graph_results.append(
    vector_stats.save_result(
        format="Parquet",
        options={
            "filename_prefix": "0210_vector_stats",
        },
    )
)
process_graph_results.append(
    vector_stats.save_result(
        format="CSV",
        options={
            "filename_prefix": "0210_vector_stats",
        },
    )
)

# Run job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)
job = multi_result.create_job()

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-script/
!rm -r output-script/

In [ ]:
results.download_files("output-script/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)